# Baseline Submission Pipeline (Cực nhanh & Không tràn RAM với bm25s)
Sử dụng thư viện `bm25s` được viết bằng C/Rust để chạy BM25 trên 4.4 triệu bài báo (3.1GB) mà không làm sập RAM 30GB của Kaggle.


In [ ]:
!pip install -q bm25s pandas pyarrow


In [ ]:
import pandas as pd
import glob
import bm25s

# 1. Quét danh sách file (Nhớ sửa lại path nếu Kaggle bung thư mục con)
corpus_files = glob.glob('/kaggle/input/r2ai-stage-3-raw-corpus-text/**/*.parquet', recursive=True)
if not corpus_files:
    print("CẢNH BÁO: KHÔNG TÌM THẤY DỮ LIỆU. Hãy kiểm tra lại đường dẫn /kaggle/input/...")

doc_ids = []
corpus_texts = []
print("1. Đang nạp toàn bộ 4.4 triệu bài báo vào RAM...")

for f in corpus_files:
    df = pd.read_parquet(f, columns=['id', 'text']).dropna(subset=['text'])
    doc_ids.extend(df['id'].astype(str).tolist())
    corpus_texts.extend(df['text'].astype(str).tolist())

print(f"Tổng số bài báo: {len(doc_ids)}")

# 2. Tokenize (Siêu tốc bằng C++)
print("2. Đang Tokenize (tách từ)...")
corpus_tokens = bm25s.tokenize(corpus_texts)

# Giải phóng RAM
del corpus_texts 

# 3. Build Index
print("3. Đang Build BM25 Index...")
retriever = bm25s.BM25()
retriever.index(corpus_tokens)
print("Hoàn tất Build Index! Đã sẵn sàng tìm kiếm.")


In [ ]:
import json

# Sửa tên thư mục này thành dataset gốc của Cuộc thi
df_queries = pd.read_parquet('/kaggle/input/TÊN-DATASET-CUỘC-THI/query.parquet')

results = []
print("4. Đang chạy tìm kiếm Inference...")

for idx, row in df_queries.iterrows():
    # Theo chuẩn BTC, id là số nguyên (integer)
    q_id = int(row['id'])
    q_text = str(row['query'])
    
    q_tokens = bm25s.tokenize([q_text])
    
    # Tìm Top 100
    doc_indices, scores = retriever.retrieve(q_tokens, k=100)
    top_k_indices = doc_indices[0]
    
    # LỖI KIỂU DỮ LIỆU CỦA BTC: doc_id TRONG ĐÁP ÁN LÀ INTEGER, KHÔNG PHẢI STRING!!!
    result_doc_ids = [int(doc_ids[i]) for i in top_k_indices]
    
    # Format nộp bài JSON của BTC
    results.append({
        "id": q_id,
        "relevant_docs": result_doc_ids,
        "relevant_chunks": []  # Có thể để rỗng nếu chỉ tìm tài liệu
    })
    
    if (idx + 1) % 10 == 0:
        print(f"Đã xử lý {idx + 1}/{len(df_queries)} truy vấn...")

with open('submission.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Đã tạo file submission.json thành công!")

import zipfile
with zipfile.ZipFile('submission.zip', 'w') as zf:
    zf.write('submission.json')
print("🎉 Đã nén thành submission.zip. HÃY NỘP FILE NÀY THÔI!")
